# 08 — Building the Gold Layer

The **gold layer** is the final tier of the medallion architecture — curated, business-ready tables designed for direct consumption by analysts, dashboards, and reporting tools.

In Notebook 07, we cleaned and standardised the silver layer into three tables:

| Silver table | Rows | Description |
| --- | --- | --- |
| `silver.pupils_cleaned` | 111 | Pupil observations — standardised labels, parsed JSON, typed columns |
| `silver.schools_cleaned` | 75 | School observations — canonical names, standardised types/phases, parsed JSON |
| `silver.contacts_cleaned` | 115 | Contact records — extracted from JSON, deduplicated, keyed by pupil + term + year |

In this notebook, we **promote** these tables to the `gold` schema with business-friendly naming (dropping the `_cleaned` suffix) and table-level documentation. If your gold promotion involves complex transformations, that’s a sign the work should have been done in silver.

### Prerequisites

This notebook **reads** from `catalog_40_copper_analyst_training.silver` and **writes** to a catalog of your choice to demonstrate table and column comments (which cannot be added to temporary views).

Use the **`catalog_name`** widget at the top to set your target catalog.

In [0]:
-- Set the session catalog to the user-provided widget value

USE CATALOG IDENTIFIER(:catalog_name);

In [0]:
-- Create the gold schema if it doesn't already exist in the selected catalog

CREATE SCHEMA IF NOT EXISTS gold;

---

## Promoting `schools_cleaned` → `gold.schools`

The schools table contains one row per school per term/year observation, with standardised school names, types, phases, status, locations, regions, Ofsted ratings, and inspection dates.

In [0]:
-- Promote schools_cleaned to gold with a descriptive table comment
-- Source: shared training catalog | Target: user's catalog (set by widget)

CREATE OR REPLACE TABLE gold.schools
COMMENT 'Gold layer: Cleaned and standardised school records. One row per school_urn per term/year. Source: silver.schools_cleaned.'
AS
SELECT *
FROM catalog_40_copper_analyst_training.silver.schools_cleaned;

SELECT * FROM gold.schools LIMIT 12;

In [0]:
-- Validate: row count should be 75, matching silver.schools_cleaned

SELECT 'gold layer' as layer
      ,COUNT(*)                                  AS row_count
      ,COUNT(DISTINCT school_urn)                 AS distinct_schools
      ,COUNT(DISTINCT CONCAT(term, '-', year))    AS distinct_periods
FROM gold.schools
UNION ALL
SELECT 'silver layer' as layer
      ,COUNT(*)                                  AS row_count
      ,COUNT(DISTINCT school_urn)                 AS distinct_schools
      ,COUNT(DISTINCT CONCAT(term, '-', year))    AS distinct_periods
FROM catalog_40_copper_analyst_training.silver.schools_cleaned;

---

## Promoting `pupils_cleaned` → `gold.pupils`

The pupils table contains one row per pupil per term/year observation, with standardised demographics (gender, ethnicity, year group), SEN status, attendance, FSM eligibility, and address fields.

In [0]:
-- Promote pupils_cleaned to gold with a descriptive table comment
-- Source: shared training catalog | Target: user's catalog (set by widget)

CREATE OR REPLACE TABLE gold.pupils
COMMENT 'Gold layer: Cleaned and standardised pupil records. One row per pupil_id per term/year. Source: silver.pupils_cleaned.'
AS
SELECT *
FROM catalog_40_copper_analyst_training.silver.pupils_cleaned;

SELECT * FROM gold.pupils LIMIT 12;

In [0]:
-- Validate: row count should be 111, matching silver.pupils_cleaned

SELECT 'gold layer' as layer
      ,COUNT(*)                                  AS row_count
      ,COUNT(DISTINCT pupil_id)                   AS distinct_pupils
      ,COUNT(DISTINCT CONCAT(term, '-', year))    AS distinct_periods
FROM gold.pupils
UNION ALL
SELECT 'silver layer' as layer
      ,COUNT(*)                                  AS row_count
      ,COUNT(DISTINCT pupil_id)                   AS distinct_pupils
      ,COUNT(DISTINCT CONCAT(term, '-', year))    AS distinct_periods
FROM catalog_40_copper_analyst_training.silver.pupils_cleaned;

---

## Promoting `contacts_cleaned` → `gold.contacts`

The contacts table contains parent/guardian details extracted from the pupil metadata JSON. It is keyed by `pupil_id`, `term`, and `year` to enable correct joins back to the pupils table for each observation period.

In [0]:
-- Promote contacts_cleaned to gold with a descriptive table comment
-- Source: shared training catalog | Target: user's catalog (set by widget)

CREATE OR REPLACE TABLE gold.contacts
COMMENT 'Gold layer: Parent/guardian contact details. Keyed by pupil_id + term + year for joins to gold.pupils. Source: silver.contacts_cleaned.'
AS
SELECT *
FROM catalog_40_copper_analyst_training.silver.contacts_cleaned;

SELECT * FROM gold.contacts LIMIT 12;

In [0]:
-- Validate: row count should be 115, matching silver.contacts_cleaned

SELECT 'gold layer' as layer
      ,COUNT(*)                                  AS row_count
      ,COUNT(DISTINCT pupil_id)                   AS distinct_pupils
      ,COUNT(DISTINCT CONCAT(term, '-', year))    AS distinct_periods
FROM gold.contacts
UNION ALL
SELECT 'silver layer' as layer
      ,COUNT(*)                                  AS row_count
      ,COUNT(DISTINCT pupil_id)                   AS distinct_pupils
      ,COUNT(DISTINCT CONCAT(term, '-', year))    AS distinct_periods
FROM catalog_40_copper_analyst_training.silver.contacts_cleaned;

---

## Gold Layer Overview

All three tables have been promoted to the `gold` schema. Here is a final cross-table summary:

In [0]:
-- Final summary: one row per gold table with key counts

SELECT 'gold.schools'  AS gold_table, COUNT(*) AS total_rows, COUNT(DISTINCT school_urn) AS distinct_entities FROM gold.schools
UNION ALL
SELECT 'gold.pupils',                  COUNT(*),              COUNT(DISTINCT pupil_id)   FROM gold.pupils
UNION ALL
SELECT 'gold.contacts',                COUNT(*),              COUNT(DISTINCT pupil_id)   FROM gold.contacts;

---

## In-Place Documentation: Table and Column Comments

Each gold table already has a **table-level comment** (set during `CREATE OR REPLACE TABLE`). Now we add **column-level comments** so that anyone browsing Catalog Explorer, a BI tool, or using `DESCRIBE TABLE` can immediately understand each field.

This is best practice for any gold layer table: the schema itself becomes the documentation.

In [0]:
-- Column comments for gold.schools

ALTER TABLE gold.schools
  ALTER COLUMN school_urn        COMMENT 'Unique Reference Number — permanent school identifier';

ALTER TABLE gold.schools
  ALTER COLUMN school_name       COMMENT 'Canonical school name — longest properly-cased variant per URN';

ALTER TABLE gold.schools
  ALTER COLUMN location          COMMENT 'Town or city where the school is located (INITCAP). Unknown if missing.';

ALTER TABLE gold.schools
  ALTER COLUMN school_type       COMMENT 'Academy, Maintained, Free School, or Unknown';

ALTER TABLE gold.schools
  ALTER COLUMN status            COMMENT 'Open, Closed, or Unknown';

ALTER TABLE gold.schools
  ALTER COLUMN phase             COMMENT 'Primary, Secondary, All-through, or Unknown';

ALTER TABLE gold.schools
  ALTER COLUMN region            COMMENT 'Geographic region (INITCAP). Unknown if missing.';

ALTER TABLE gold.schools
  ALTER COLUMN ofsted_rating     COMMENT 'Latest Ofsted rating as string (e.g. Outstanding, Good). Unknown if missing.';

ALTER TABLE gold.schools
  ALTER COLUMN last_inspection   COMMENT 'Date of last Ofsted inspection. NULL if not available.';

ALTER TABLE gold.schools
  ALTER COLUMN pupil_premium_pct COMMENT 'Percentage of pupils eligible for pupil premium funding. NULL if not available.';

ALTER TABLE gold.schools
  ALTER COLUMN term              COMMENT 'Academic term of this observation (e.g. Autumn, Spring, Summer)';

ALTER TABLE gold.schools
  ALTER COLUMN year              COMMENT 'Academic year of this observation (e.g. 2024)';

You can then view the table with the comments in-place by looking in the Catalog for your table. This can be done from the 'Catalog' page, or from the left hand sidebar of the notebook. 

![Table and column comments on gold.schools](../images/data-modelling-comment-schools.png)

To find your table in the lefthand sidebar click the button with the three shapes (or press `Alt` + `Ctrl` + `C`) search for your table then press the three dots to the right of it and click 'Open in Catalog Explorer'.

![Catalog explorer](../images/catalog_explorer.png)

In [0]:
-- Column comments for gold.pupils

ALTER TABLE gold.pupils
  ALTER COLUMN pupil_id          COMMENT 'Unique pupil identifier (e.g. PUP001)';

ALTER TABLE gold.pupils
  ALTER COLUMN first_name        COMMENT 'Pupil first name (INITCAP standardised)';

ALTER TABLE gold.pupils
  ALTER COLUMN last_name         COMMENT 'Pupil last name (INITCAP standardised)';

ALTER TABLE gold.pupils
  ALTER COLUMN gender            COMMENT 'Male, Female, or Unknown';

ALTER TABLE gold.pupils
  ALTER COLUMN date_of_birth     COMMENT 'Date of birth parsed from mixed formats (ISO/UK/US). NULL if unparseable.';

ALTER TABLE gold.pupils
  ALTER COLUMN school_urn        COMMENT 'FK to gold.schools — Unique Reference Number of the attending school';

ALTER TABLE gold.pupils
  ALTER COLUMN year_group        COMMENT 'Year 6, Year 7, Year 8, Year 9, or Unknown';

ALTER TABLE gold.pupils
  ALTER COLUMN ethnicity         COMMENT 'Standardised ethnicity label (Title Case). Unknown if missing.';

ALTER TABLE gold.pupils
  ALTER COLUMN term              COMMENT 'Academic term of this observation (e.g. Autumn, Spring, Summer)';

ALTER TABLE gold.pupils
  ALTER COLUMN year              COMMENT 'Academic year of this observation (e.g. 2024)';

ALTER TABLE gold.pupils
  ALTER COLUMN sen_status        COMMENT 'EHCP, SEN Support, None, or Unknown';

ALTER TABLE gold.pupils
  ALTER COLUMN fsm_eligible      COMMENT 'Whether the pupil is eligible for Free School Meals. NULL if not available.';

ALTER TABLE gold.pupils
  ALTER COLUMN attendance_pct    COMMENT 'Attendance percentage (0–100). NULL if not available.';

ALTER TABLE gold.pupils
  ALTER COLUMN address_line1     COMMENT 'First line of pupil home address. NULL if not available.';

ALTER TABLE gold.pupils
  ALTER COLUMN address_city      COMMENT 'City of pupil home address. NULL if not available.';

ALTER TABLE gold.pupils
  ALTER COLUMN address_postcode  COMMENT 'Postcode of pupil home address. NULL if not available.';

![Table and column comments on gold.pupils](../images/data-modelling-comments-pupils.png)

In [0]:
-- Column comments for gold.contacts

ALTER TABLE gold.contacts
  ALTER COLUMN pupil_natural_key COMMENT 'Natural key combining pupil name and DOB — used during deduplication';

ALTER TABLE gold.contacts
  ALTER COLUMN pupil_id          COMMENT 'FK to gold.pupils — pupil this contact belongs to';

ALTER TABLE gold.contacts
  ALTER COLUMN term              COMMENT 'Academic term — part of composite key for joins to gold.pupils';

ALTER TABLE gold.contacts
  ALTER COLUMN year              COMMENT 'Academic year — part of composite key for joins to gold.pupils';

ALTER TABLE gold.contacts
  ALTER COLUMN parent_name       COMMENT 'Parent or guardian full name';

ALTER TABLE gold.contacts
  ALTER COLUMN phone             COMMENT 'Contact phone number. NULL if not provided.';

ALTER TABLE gold.contacts
  ALTER COLUMN email             COMMENT 'Contact email address. NULL if not provided.';

ALTER TABLE gold.contacts
  ALTER COLUMN address_line1     COMMENT 'First line of contact address. NULL if not provided.';

ALTER TABLE gold.contacts
  ALTER COLUMN address_city      COMMENT 'City of contact address. NULL if not provided.';

ALTER TABLE gold.contacts
  ALTER COLUMN address_postcode  COMMENT 'Postcode of contact address. NULL if not provided.';

![Table and column comments on gold.contacts](../images/data-modelling-comments-contacts.png)

---

## Module Recap: Data Modelling and Standardisation

### The pipeline we built

```
Source Files  →  Bronze (raw)  →  Silver (clean)  →  Gold (curated)
```

| Step | Notebook | What we did |
| --- | --- | --- |
| **Preparation overview** | 01 — Data Preparation | Previewed common data quality issues (duplicates inflating counts, inconsistent labels, orphaned foreign keys) and outlined the end-to-end cleaning workflow |
| **Medallion architecture** | 02 — Medallion Architecture | Introduced the bronze → silver → gold layered pattern, how the DfE implements it, and the role of each layer |
| **Combining snapshots** | 03 — Combining Snapshots | Dynamically combined termly bronze extracts into `silver.schools_combined` and `silver.pupils_combined` using `INFORMATION_SCHEMA` and procedural SQL |
| **Schema reconciliation** | 04 — Reconciling Schema Drift | Unified renamed columns across snapshots (e.g. `city`/`local_authority`, `school_name`/`establishment_name`) using `COALESCE` |
| **Exact deduplication** | 05 — Understanding Your Data | Removed exact duplicate rows using `CONCAT_WS` + `ROW_NUMBER`, profiled cardinality and referential integrity |
| **Natural key deduplication** | 06 — Removing Duplicates | Resolved natural key conflicts (same entity + term + year with different values) using ranked business rules and `ROW_NUMBER`, preserving discarded rows in `audit_*` tables |
| **Cleaning & standardisation** | 07 — Cleaning the Silver Layer | Standardised labels (lookup tables), fixed casing (`INITCAP`), parsed JSON, cast types, handled missing data (`COALESCE(..., 'Unknown')` for categorical, `NULL` for non-categorical), enriched from audit tables, extracted contacts |
| **Gold promotion** | 08 — Building the Gold Layer | Promoted clean tables to the gold schema with business-friendly names and table-level documentation |
| **In-place documentation** | 08 — Building the Gold Layer | Added table and column comments so the schema is self-describing in Catalog Explorer and BI tools |

---
## Clean up

The gold tables are example data with no real meaning. Once you’ve finished inspecting them, remove them along with the `gold` schema using the cell below.

To run it: click into the cell, press `Ctrl+K` then `Ctrl+U` to uncomment, then `Ctrl+Enter` to execute.

> **Tip:** `Ctrl+K` then `Ctrl+C` comments out highlighted lines.

In [0]:
-- DROP SCHEMA IF EXISTS gold CASCADE;